<a href="https://colab.research.google.com/github/giacomomolinari/liar-fake-news-detector/blob/main/models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification Models

## Imports

In [ ]:
%pip install torchmetrics==1.9.0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import transformers
import random
import copy
import os

from datasets import load_dataset
from google.colab import drive
from pathlib import Path

from imblearn.under_sampling import RandomUnderSampler


from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.metrics import precision_score, recall_score

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from datasets import Dataset, load_dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchmetrics.classification import Accuracy

from transformers import TrainingArguments, DataCollatorWithPadding, Trainer, BertForSequenceClassification

## 1. Getting the data

The notebook assumes that the LIAR dataset is available on your Google Drive at the path defined below. You can download the dataset from [William Yang Wang's website](https://sites.cs.ucsb.edu/~william/data/liar_dataset.zip).

In [ ]:
# replace this with the path to your dataset
DATASET_PATH = Path("/content/drive/MyDrive/datasets/liar_dataset/")

In [ ]:
drive.mount('/content/drive')

In [ ]:
columns= ["ID", "Label", "Statement", "Subjects", "Speaker", "SpeakerJob", "State", "Party",
          "HistBarelyTrue", "HistFalse", "HistHalfTrue", "HistMostTrue", "HistPantsFire", "Context"]

In [ ]:
df_train = pd.read_csv(DATASET_PATH / "train.tsv", sep="\t", header=0, names=columns)
df_valid = pd.read_csv(DATASET_PATH / "valid.tsv", sep="\t", header=0, names=columns)
df_test = pd.read_csv(DATASET_PATH / "test.tsv", sep="\t", header=0, names=columns)

In [ ]:
df_train.head()

## 2. Preprocessing

### 2.1 Handling NaN values

As discussed in the eda notebook, only two columns have significant amounts of NaN values. I will remove these columns from the dataset to begin with.

In [ ]:
df_train.info()

In [ ]:
df_train_clean = df_train.drop(["SpeakerJob", "State"], axis=1)
df_train_clean.info()

For `NaN` values in the `Context` column, we will replace them with "no context provided."

In [ ]:
df_train_clean.loc[df_train_clean["Context"].isna(), "Context"] = "no context provided."

In [ ]:
df_train_clean[df_train_clean.isna().any(axis=1)]

Only two rows with NaN values remain, and these have all values NaN except for the statement. I will drop these for simplicity.

In [ ]:
df_train_clean = df_train_clean.dropna()

Let's define a convenience function to apply the same changes to the validation and test dataframes

In [ ]:
def clean_dataset(df):
  df_clean = df.drop(["SpeakerJob", "State"], axis=1)
  df_clean.loc[df_clean["Context"].isna(), "Context"] = "no context provided."
  df_clean = df_clean.dropna()
  return df_clean

In [ ]:
df_valid_clean = clean_dataset(df_valid)

df_valid_clean.info()

### 2.2 Selecting and Aggregating Features

As discussed in the EDA notebook, we will aggregate the `pants-fire` label with the `false` label to ensure the model is only learning about statement truthfulness, rather than learning possibly normative features like whether a lie is obvious, brazen, shocking or extravagant.

In [ ]:
df_train_reduced = df_train_clean.copy()

In [ ]:
df_train_reduced.loc[df_train_reduced["Label"] == "pants-fire", "Label"] = "false"

Then we will drop the "speaker history" features which contain the count of entries with each of the labels for the statement's speaker, and thus introduce data leak risks.

In [ ]:
df_train_reduced = df_train_reduced.drop(["HistBarelyTrue", "HistFalse", "HistHalfTrue", "HistMostTrue", "HistPantsFire"], axis=1)

In [ ]:
df_train_reduced.head()

Again we define a utility function to perform this transformation.

In [ ]:
def aggregate_and_drop(df):
  df_res = df.copy()
  df_res.loc[df_res["Label"] == "pants-fire", "Label"] = "false"
  df_res = df_res.drop(["HistBarelyTrue", "HistFalse", "HistHalfTrue", "HistMostTrue", "HistPantsFire"], axis=1)
  return df_res

In [ ]:
df_valid_reduced = aggregate_and_drop(df_valid_clean)

df_valid_reduced.head()

### 2.3 Multi-hot encoding of Subject feature

The Subject feature is a list of tags, so we will encode it by creating a new feature for each (sufficiently common) tag, which is 1 iff the sample in question has that subject tag.

In [ ]:
subject_set = set([])
subject_series = []
for subject in df_train_reduced["Subjects"]:
  if not isinstance(subject, str):
    print(subject)
  subject = subject.replace(", ", ",")
  split = subject.split(",")
  for s in split:
    if s not in subject_set:
      subject_set.add(s)
    subject_series.append(s)

In [ ]:
subject_series = pd.Series(subject_series)
counts = subject_series.value_counts()
counts.count(), counts[counts > 100].count()

In [ ]:
common_subjects = set(counts[counts > 100].index)

there are 59 subjects which appear over 100 times, while all others appear less than 100 times each. I will aggregate these infrequent subject tags into a new tag "other"

In [ ]:
df_train_listsubject = df_train_reduced.copy()
df_train_listsubject["Subjects"] = (df_train_listsubject["Subjects"].apply(lambda x: x.replace(", ", ",").split(","))
  .apply(lambda x: [s if s in common_subjects else "other" for s in x])
  .apply(lambda x: list(set(x)))) # remove duplicates

In [ ]:
df_train_listsubject.shape

In [ ]:
df_exploded = df_train_listsubject.explode("Subjects")
df_exploded.head()

In [ ]:
one_hot = pd.get_dummies(df_exploded['Subjects'])

In [ ]:
one_hot.shape

In [ ]:
multi_hot = one_hot.groupby(one_hot.index).sum()
multi_hot.shape

In [ ]:
df_train_multihot = df_train_reduced.merge(multi_hot, left_index=True, right_index=True).drop("Subjects", axis=1)
df_train_multihot.shape


To the original 7 features we added 60 subject features (59 for common subjects, 1 for "other"), and then removed the pre-existing "Subjects" feature, for a total of 66 features in the final dataset.

Again we define a utility function to more easily perform this transformation on the validation and test datasets.

In [ ]:
def multi_hot_encode(df, subject_set):
  df_listsubject = df.copy()
  df_listsubject["Subjects"] = (df_listsubject["Subjects"].apply(lambda x: x.replace(", ", ",").split(","))
    .apply(lambda x: [s if s in subject_set else "other" for s in x])
    .apply(lambda x: list(set(x)))) # remove duplicates

  df_exploded = df_listsubject.explode("Subjects") # add new rows for each value in "Subjects" list, copying other fields

  one_hot = pd.get_dummies(df_exploded['Subjects']) # one-hot encode all possible "Subject" values
  multi_hot = one_hot.groupby(one_hot.index).sum()  # group back by index to restore origina number of columns
  return multi_hot.merge(df_listsubject.drop("Subjects", axis=1), left_index=True, right_index=True)

In [ ]:
df_valid_reduced.shape

In [ ]:
df_valid_multihot = multi_hot_encode(df_valid_reduced, common_subjects)
df_valid_multihot.shape

### 2.4 Statement Length

I will remove all statements with over 40 words from the dataset. As shown in the EDA notebook, this adds up to about 100 statements, some of which are hundreds of words long, well above the dataset mode of 15. By removing these long statements we reduce the amount of padding needed to ensure all sequences processed by the RNN have the same length, which can improve performance.

In [ ]:
df_train_preprocessed = df_train_multihot.copy()
df_train_preprocessed["StatementLength"] = df_train_preprocessed["Statement"].apply(lambda x: len(x.split(" ")))
df_train_preprocessed[["Statement", "StatementLength"]].head()

In [ ]:
df_train_preprocessed.shape

In [ ]:
df_train_preprocessed = df_train_preprocessed[df_train_preprocessed["StatementLength"] < 40]
df_train_preprocessed.shape

As usual we define a utility function for this transformation.

In [ ]:
def drop_statements_longer_than(df, maxLength):
  df_res = df.copy()
  df_res["StatementLength"] = df_res["Statement"].apply(lambda x: len(x.split(" ")))
  df_res = df_res[df_res["StatementLength"] < maxLength]
  return df_res

In [ ]:
df_valid_preprocessed = drop_statements_longer_than(df_valid_multihot, 40)
df_valid_preprocessed.shape

## 3. Baseline Model 1: Metadata-only classifier

As a first baseline let's build a model that tries to classify statements based only on the available metadata: the name of the speaker, their political affiliation, and the topics touched on by the statement. I will also include the statement length, which is a derived piece of metadata and may potentially be helpful.

### 3.1 Preparing the data

We need a little bit more preprocessing before the metadata is ready to be used by a ML classifier. In particular, the remaining categorical features (`Speaker` and `Party`) must be converted to numerical ones, and the numerical feature `StatementLength` must be normalized.

#### 3.1.1 Normalizing StatementLength

In [ ]:
subjects_set_list = list(common_subjects.union({"other"}))
meta_features = ["Speaker", "Party", "StatementLength"] + subjects_set_list

X_train_meta = df_train_preprocessed[meta_features]
y_train_meta = df_train_preprocessed["Label"]

X_train_meta.head()

In [ ]:
min_max_scaler = MinMaxScaler(feature_range=(0, 1))
X_train_meta[["StatementLength"]] = min_max_scaler.fit_transform(X_train_meta[["StatementLength"]])
X_train_meta.head()

In [ ]:
statement_length_normalized = X_train_meta["StatementLength"]
statement_length_normalized.describe()

#### 3.1.2. Encoding Speaker

In [ ]:
speaker_counts = X_train_meta["Speaker"].value_counts()
speaker_counts

Clearly there are too many speakers for one-hot encoding. But for many of these speakers, they appear too infrequently for the model to actually be able to learn much about them. So let's look at how many speakers appear frequently.

In [ ]:
speaker_counts[speaker_counts>50].count()

Interestingly, only 20 speakers appear over 50 times. So we will aggregate all other speakers to a new value "other", and then one-hot encode this feature to generate 21 more features (and then remove the existing `Speaker` feature)

In [ ]:
common_speakers = set(speaker_counts[speaker_counts>50].index)

X_train_speaker_encoded = X_train_meta.copy()
X_train_speaker_encoded["Speaker"] = X_train_meta["Speaker"].apply(lambda x: x if x in common_speakers else "other")
X_train_speaker_encoded = pd.get_dummies(X_train_speaker_encoded, columns=["Speaker"])
X_train_speaker_encoded.head()

#### 3.1.3 Encoding Party

We can one-hot encode the `Party` feature in a similar way.

In [ ]:
party_counts = X_train_meta["Party"].value_counts()
party_counts[party_counts>50]

Again it makes sense to aggregate parties that are too infrequent to be actually learned by the model.

In [ ]:
common_parties = set(party_counts[party_counts>50].index)
X_train_party_encoded = X_train_speaker_encoded.copy()

X_train_party_encoded["Party"] = X_train_speaker_encoded["Party"].apply(lambda x: x if x in common_parties else "other")
X_train_party_encoded = pd.get_dummies(X_train_party_encoded, columns=["Party"])
X_train_party_encoded.head()

In [ ]:
X_train_party_encoded.shape

Finally we convert the boolean values to integers for numerical processing and consistency.

In [ ]:
X_train_meta = X_train_party_encoded.copy()

X_statement_length = X_train_meta["StatementLength"].copy()
X_train_meta = X_train_meta.map(lambda x: int(x))
X_train_meta["StatementLength"] = X_statement_length

X_train_meta.head()

As before we define a function that performs all these transformations on a dataset.

In [ ]:
def meta_preprocessing(df):
  df_res = df.copy()

  min_max_scaler = MinMaxScaler(feature_range=(0, 1))
  df_res["StatementLength"] = min_max_scaler.fit_transform(df[["StatementLength"]])

  df_res["Speaker"] = df_res["Speaker"].apply(lambda x: x if x in common_speakers else "other")
  df_res = pd.get_dummies(df_res, columns=["Speaker"])

  df_res["Party"] = df_res["Party"].apply(lambda x: x if x in common_parties else "other")
  df_res = pd.get_dummies(df_res, columns=["Party"])

  df_statement_length = df_res["StatementLength"].copy()
  df_res = df_res.map(lambda x: int(x))
  df_res["StatementLength"] = df_statement_length

  return df_res

In [ ]:
X_valid_meta = df_valid_preprocessed[meta_features]
y_valid_meta = df_valid_preprocessed["Label"]

X_valid_meta.head()

In [ ]:
X_valid_meta = meta_preprocessing(X_valid_meta)
X_valid_meta.head()

### 3.2 Building the Models

#### 3.2.1 Defining the evaluation measures

We will evaluate our models by first converting the labels into integers and then calculating the MAE. This preserves the intuitive ordering that is intrinsic in these truthfulness labels. For example, if a statement is almost-true, it seems better to have classified it as true than to have classified it as false (although one could plausibly wish to specify just how much better that is, I will for simplicity just take this to be specified by the MAE metric).

I will also evaluate our model using the usual accuracy metric, for comparison and more interpretable results.

In [ ]:
label_to_int = {"false": 0, "barely-true": 1, "half-true": 2, "mostly-true": 3, "true": 4}

y_train_meta_int = y_train_meta.apply(lambda x: label_to_int[x])
y_valid_meta_int = y_valid_meta.apply(lambda x: label_to_int[x])

In [ ]:
scoring_rule = "accuracy"
scoring_rule_ordinal = "neg_mean_absolute_error"

#### 3.2.2 Simple Baseline Classifiers

As a sanity check and starting baseline, it's useful to define a **Majority Class Classifier**. This is a classifier which just predicts the most frequent class regardless of input. Its performance on the task provides a performance floor which our models should be able to clear if they have learned anything at all

In [ ]:
y_mode = y_train_meta_int.mode()[0]
y_mode

In [ ]:
def majority_class_classifier(X, y):
  return np.ones((len(X), 1))* y.mode()[0]

In [ ]:
mean_absolute_error(y_valid_meta_int, majority_class_classifier(X_valid_meta, y_train_meta_int))

In [ ]:
accuracy_score(y_valid_meta_int, majority_class_classifier(X_valid_meta, y_train_meta_int))

As expected this has an accuracy of around 30%, as around 27% of statements in the training dataset are false.

#### 3.2.3 Tree classifier

We use a simple tree classifier, limiting depth to avoid it catastrophically overfitting the training set.

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42, max_depth=7)

tree_model

In [ ]:
tree_model_cv = -cross_val_score(tree_model, X_train_meta, y_train_meta_int, scoring=scoring_rule_ordinal, cv=10)

tree_model_cv.mean()

In [ ]:
tree_model.fit(X_train_meta, y_train_meta_int)

In [ ]:
mean_absolute_error(y_valid_meta_int, tree_model.predict(X_valid_meta))

In [ ]:
accuracy_score(y_valid_meta_int, tree_model.predict(X_valid_meta))

This does noticeably better than the majority class predictor in MAE and slightly better in accuracy. To check that it's in fact not overfitting, let's look at the accuracy on training set

In [ ]:
accuracy_score(y_train_meta_int, tree_model.predict(X_train_meta))

About the same as validation accuracy, so no sign of strong overfit.

In [ ]:
y_pred = tree_model.predict(X_valid_meta)
pd.Series(y_pred).value_counts()

We can see here that the model almost degenerates to the majority classifier. The model is not really learning from the features - rather it is learning that False is the most probable label, and hedging with some almost-true predictions.

#### 3.2.4 Random forest classifier

Let's try a slightly more powerful model, with slightly more fine-tuning.

In [ ]:
forest_model = RandomForestClassifier(random_state=42)

forest_model_cv = cross_val_score(forest_model, X_train_meta, y_train_meta_int, scoring=scoring_rule, cv=5)

forest_model_cv.mean()

In [ ]:
forest_model.fit(X_train_meta, y_train_meta_int)

In [ ]:
mean_absolute_error(y_valid_meta_int, forest_model.predict(X_valid_meta))

In [ ]:
accuracy_score(y_valid_meta_int, forest_model.predict(X_valid_meta))

In [ ]:
accuracy_score(y_train_meta_int, forest_model.predict(X_train_meta))

Much like for the tree classifier, the unconstrained version of RandomForest heavily overfits the training data. So let's look for some regularization hyperparameter values that improve this.

In [ ]:
parameter_grid = [
    {
        "max_depth": [10, 15, 20, 25, 30],
        "max_features": [10, 20, 30, 40, 50]
    }
]

In [ ]:
forest_cls = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(forest_cls, parameter_grid, cv=5, scoring=scoring_rule)
grid_search.fit(X_train_meta, y_train_meta_int)
grid_search.best_params_

In [ ]:
forest_model = RandomForestClassifier(random_state=42, max_depth=10, max_features=10)


forest_model.fit(X_train_meta, y_train_meta_int)

In [ ]:
mean_absolute_error(y_valid_meta_int, forest_model.predict(X_valid_meta))

In [ ]:
accuracy_score(y_valid_meta_int, forest_model.predict(X_valid_meta))

In [ ]:
accuracy_score(y_train_meta_int, forest_model.predict(X_train_meta))

In [ ]:
pd.Series(y_pred).value_counts()

Similar problem as the tree model, the RandomForest almost degenerates to a majority classifier, with a dominating cluster of predictions on "False", and a smaller cluster of predictions on "Almost True"

#### 3.2.5 Reducing the feature space

In [ ]:
X_train_meta_reduced = X_train_meta.drop(common_subjects, axis=1)
X_train_meta_reduced = X_train_meta_reduced.drop(["Party_" + x for x in common_parties.union(set(['other']))], axis=1)
X_train_meta_reduced.head()

X_valid_meta_reduced = X_valid_meta.drop(common_subjects, axis=1)
X_valid_meta_reduced = X_valid_meta_reduced.drop(["Party_" + x for x in common_parties.union(set(['other']))], axis=1)
X_valid_meta_reduced.head()

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42, max_depth=11)

tree_model_cv = -cross_val_score(tree_model, X_train_meta_reduced, y_train_meta_int, scoring=scoring_rule_ordinal, cv=10)

tree_model_cv.mean()

In [ ]:
tree_model.fit(X_train_meta_reduced, y_train_meta_int)

In [ ]:
mean_absolute_error(y_valid_meta_int, tree_model.predict(X_valid_meta_reduced)), accuracy_score(y_valid_meta_int, tree_model.predict(X_valid_meta_reduced))

In [ ]:
accuracy_score(y_train_meta_int, tree_model.predict(X_train_meta_reduced))

In [ ]:
y_pred = tree_model.predict(X_valid_meta_reduced)
pd.Series(y_pred).value_counts()

Still overfitting unless we let it deteriorate to majority classifier.

#### 3.2.6 Support Vector Classifier

Let's try using a support vector classifier, as SVCs tend to work well in high-dimensional spaces.

In [ ]:
svm_clf = SVC(random_state=42)

svm_clf_cv = -cross_val_score(svm_clf, X_train_meta, y_train_meta_int, scoring=scoring_rule_ordinal, cv=5)

svm_clf_cv.mean()

This is not much better than previous examples.

In [ ]:
svm_clf.fit(X_train_meta, y_train_meta_int)

In [ ]:
pd.Series([mean_absolute_error(y_valid_meta_int, svm_clf.predict(X_valid_meta)),
 accuracy_score(y_valid_meta_int, svm_clf.predict(X_valid_meta)),
 accuracy_score(y_train_meta_int, svm_clf.predict(X_train_meta))], index=["MAE", "AccuracyValid", "AccuracyTest"])

In [ ]:
pd.Series(svm_clf.predict(X_valid_meta)).value_counts()

The model is overfitting a little, and as we attempt to reduce this overfit we observe the same sort of deterioration to majority classifier.

#### 3.2.7 Rebalancing class frequencies

In [ ]:
UnderSampler = RandomUnderSampler(random_state=42, replacement=True)

X_under, y_under = UnderSampler.fit_resample(X_train_meta, y_train_meta_int)

X_under_valid, y_under_valid = UnderSampler.fit_resample(X_valid_meta, y_valid_meta_int)

y_under.value_counts(), y_under_valid.value_counts()

After undersampling all 5 classes have the exact same frequency in the dataset. Hence our baseline majority classifier will have exactly 20% accuracy.

In [ ]:
accuracy_score(y_under_valid, majority_class_classifier(X_under_valid, y_under))

In [ ]:
X_under.head()

In [ ]:
X_under_reduced = X_under.drop(common_subjects, axis=1)
X_under_reduced = X_under.drop(["Party_" + x for x in common_parties.union(set(['other']))], axis=1)
X_under_reduced.head()

X_under_valid_reduced = X_under_valid.drop(common_subjects, axis=1)
X_under_valid_reduced = X_under_valid.drop(["Party_" + x for x in common_parties.union(set(['other']))], axis=1)
X_under_valid_reduced.head()

In [ ]:
svm_clf = SVC(random_state=42, C=10)

svm_clf_cv = -cross_val_score(svm_clf, X_under_reduced, y_under, scoring=scoring_rule_ordinal, cv=5)

svm_clf_cv.mean()

In [ ]:
svm_clf.fit(X_under, y_under)

In [ ]:
pd.Series([mean_absolute_error(y_under, svm_clf.predict(X_under)),
 accuracy_score(y_under_valid, svm_clf.predict(X_under_valid)),
 accuracy_score(y_under, svm_clf.predict(X_under))], index=["MAE", "AccuracyValid", "AccuracyTrain"])

In [ ]:
pd.Series(svm_clf.predict(X_under_valid)).value_counts()

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42, max_depth=15)

tree_model_cv = -cross_val_score(tree_model, X_under, y_under, scoring=scoring_rule_ordinal, cv=10)

tree_model_cv.mean()

In [ ]:
tree_model.fit(X_under, y_under)

In [ ]:
pd.Series([mean_absolute_error(y_under, tree_model.predict(X_under)),
 accuracy_score(y_under_valid, tree_model.predict(X_under_valid)),
 accuracy_score(y_under, tree_model.predict(X_under))], index=["MAE", "AccuracyValid", "AccuracyTest"])

In [ ]:
forest_model = RandomForestClassifier(random_state=42, max_depth=10, n_estimators=1000)

forest_model.fit(X_under, y_under)

In [ ]:
forest_model_cv = -cross_val_score(forest_model, X_under, y_under, scoring=scoring_rule_ordinal, cv=10)

forest_model_cv.mean()

In [ ]:
pd.Series([mean_absolute_error(y_under, forest_model.predict(X_under)),
 accuracy_score(y_under_valid, forest_model.predict(X_under_valid)),
 accuracy_score(y_under, forest_model.predict(X_under))], index=["MAE", "AccuracyValid", "AccuracyTest"])

This model reaches over 28% in validation accuracy, which is actually noticeably better than the majority model now that all labels have been rebalanced by undersampling. After rebalancing, the baseline majority classifier only has 20% accuracy, so this is a >8% improvement. Furthermore, the accuracy achieved by this model is comparable to that achieved in [William Wang's paper](https://https://aclanthology.org/P17-2067/) which introduced the LIAR dataset, when evaluating models that use metadata only.

The model is still overfitting, but validation accuracy seems to decrease too when introducing more regularization.

In [ ]:
pd.Series(forest_model.predict(X_under_valid)).value_counts()

We can see that, while the model is slightly skewed towards predicting falsehood still, the distribution of predictions is far more balanced now.

## 4. Baseline Model 2: Text-only classifier

### 4.1 Preparing the data

In [ ]:
statements_train = list(df_train_preprocessed["Statement"].apply(lambda x: x.lower()))
statements_valid = list(df_valid_preprocessed["Statement"].apply(lambda x: x.lower()))

statements_train[:3]

### 4.2 Tokenizer

I will use a pre-trained WordPiece tokenizer extracted from a BERT model.

In [ ]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
bert_encoding = bert_tokenizer(statements_train, padding=True, truncation=True, max_length=80, return_tensors="pt")

We can plot the length (in tokens) of each sequence by plotting the sum of the attention masks for each token.

In [ ]:
pd.Series([attention_mask.sum() for attention_mask in bert_encoding["attention_mask"]]).apply(lambda x: int(x)).plot.hist()

### 4.3 Building the Datasets and DataLoaders

First we convert our Pandas dataframes into Dataset objects

In [ ]:
label_to_int = {"false": 0, "barely-true": 1, "half-true": 2, "mostly-true": 3, "true": 4}


df_train_textonly = df_train_preprocessed[["Statement", "Label"]].copy()
df_train_textonly["Label"] = df_train_preprocessed["Label"].apply(lambda x: label_to_int[x])
df_train_textonly["Statement"] = df_train_preprocessed["Statement"].apply(lambda x: x.lower())


df_valid_textonly = df_valid_preprocessed[["Statement", "Label"]].copy()
df_valid_textonly["Label"] = df_valid_preprocessed["Label"].apply(lambda x: label_to_int[x])
df_valid_textonly["Statement"] = df_valid_preprocessed["Statement"].apply(lambda x: x.lower())

df_train_textonly.head()

In [ ]:
ds_train = Dataset.from_pandas(df_train_textonly)
ds_valid = Dataset.from_pandas(df_valid_textonly)

ds_train

Then we can use DataLoaders to tokenize the statements as they are passed to the model.

In [ ]:
def collate_fn(batch, tokenizer = bert_tokenizer):
  statements = [x["Statement"] for x in batch]
  labels = [[x["Label"]] for x in batch]

  encodings = tokenizer(statements, padding=True, truncation=True, max_length=80, return_tensors="pt")

  labels = torch.tensor(labels, dtype=torch.int64)

  return encodings, labels

In [ ]:
batch_size = 128

text_train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
text_valid_loader = DataLoader(ds_valid, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

### 4.4 Building the Models

#### 4.4.1 Untrained Model (Training from scratch)

Let's define a class for our model

In [ ]:
class LieDetector_TextOnly(nn.Module):
  def __init__(self, vocab_size, n_layers=2, embed_dim = 128, hidden_dim=64, pad_id=0, dropout=0.2):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)

    self.output = nn.Linear(hidden_dim, 5) # output class logits

  def forward(self, encodings):
    embeddings = self.embedding(encodings["input_ids"])
    _outputs, hidden_states = self.gru(embeddings)
    return self.output(hidden_states[-1])

Then let's define training and evaluation functions, using the GPU for training.

In [ ]:
if torch.cuda.is_available():
    device="cuda"
elif torch.backends.mps.is_available:
    device="mps"
else:
    device="cpu"

device

In [ ]:
# Evaluation function (not using torchmetrics)
def evaluate(model, data_loader, criterion):
  model.eval()
  total_loss = 0
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)

      y_batch = y_batch.squeeze(1)

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
  return total_loss/len(data_loader)

In [ ]:
# General training function
def train(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
  for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      y_batch = y_batch.squeeze(1)

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    eval_loss = evaluate(model, valid_loader, criterion)
    print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {mean_loss:.4f}, Validation Loss: {eval_loss:.4f}")

In [ ]:
torch.manual_seed(42)

In [ ]:
model = LieDetector_TextOnly(vocab_size=bert_tokenizer.vocab_size, pad_id=bert_tokenizer.pad_token_id).to(device)


We compute class frequencies on the non-rebalanced dataset to use the weights in scoring.

In [ ]:
class_counts = np.array(df_train_textonly.groupby("Label").count().values.flatten())
class_counts

In [ ]:
class_weights = 1.0/class_counts                                  # weight inversely proportional to number of occurrences
class_weights = torch.tensor(class_weights/ class_weights.sum())  # normalize

class_weights = class_weights.float().to(device)
class_weights

In [ ]:
learning_rate = 0.0005
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
train(model,  optimizer=optimizer, criterion=criterion, train_loader = text_train_loader, valid_loader = text_valid_loader, n_epochs=25)

The model seems to be overfitting to an extreme degree, where the validation performance actually gets worse and worse as the model fits to the training dataset.



Let's write a simple utility function to give us an intuitive sense of how the model performs on a few validation statements

In [ ]:
def sample_validation_results(model, validation_set, sample_size):
  valid_length = len(validation_set)
  statement_list = list(validation_set["Statement"])
  label_list = list(validation_set["Label"])
  sample = random.sample(range(0, valid_length), sample_size)

  df = pd.DataFrame(columns=["Statement", "Label", "Prediction"])

  for entry_idx in sample:
    statement = statement_list[entry_idx]
    label = label_list[entry_idx]

    tokenized_statement = bert_tokenizer(statement, padding=True, truncation=True, max_length=80, return_tensors="pt")

    model.eval()
    with torch.no_grad():
      prediction = model(tokenized_statement.to(device)).argmax(dim=-1).item()
      df.loc[len(df)] = [statement, label, prediction]
  return df

In [ ]:
sample_validation_results(model, ds_valid, 10)

This clearly shows that the model is performing very poorly. The likely reason for this poor performance is that the model just doesn't have enough information available in the dataset to learn reasonably informative/meaningful embeddings of the tokens, which in turn are necessary for determining whether the statements are true or false. A better approach would be to reuse the pre-trained embedding layer of an existing model, which can represent tokens into reasonably informative embeddings, and then focus our model training on determining degree of truthfulness from these embeddings.

#### 4.4.2 Using pretrained embeddings

In [ ]:
class LieDetector_TextOnly_PretrainedEmbeds(nn.Module):
  def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64, pad_id=0, dropout=0.2):
    super().__init__()
    weights = pretrained_embeddings.weight.data

    self.embedding = nn.Embedding.from_pretrained(weights, freeze=True)
    embed_dim = weights.shape[-1]

    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)

    self.output = nn.Linear(hidden_dim, 5) # output class logits

  def forward(self, encodings):
    embeddings = self.embedding(encodings["input_ids"])
    _outputs, hidden_states = self.gru(embeddings)
    return self.output(hidden_states[-1])

Let's get the entire bert model and then pass the embedding weights to our class constructor.

In [ ]:
bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")

In [ ]:
model_pretrained_embeds = LieDetector_TextOnly_PretrainedEmbeds(bert_model.embeddings.word_embeddings).to(device)

In [ ]:
learning_rate = 0.0005
optimizer = torch.optim.Adam(model_pretrained_embeds.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
train(model_pretrained_embeds,  optimizer=optimizer, criterion=criterion, train_loader = text_train_loader, valid_loader = text_valid_loader, n_epochs=40)

Unlike before, we can see here some meaningful learning in the first 20 or so epochs before the model starts overfitting the training set. This shows that using Bert's pretrained embeddings does in fact help the model distinguish between different degrees of truthfulness.

It makes sense here to implement early stopping in our training loop so that we can stop the training before the model starts overfitting.

In [ ]:
# Early stopping class
class EarlyStopping:
  def __init__(self, patience=10, min_delta=0.0):
    self.patience = patience
    self.min_delta = min_delta
    self.counter = 0
    self.best_score = None
    self.best_weigths = None

  def step(self, val_loss, model):
    #Debug
    best_score_str = f"{self.best_score:.6f}" if self.best_score is not None else "None"
    print(f"val_loss: {val_loss:.6f}, best_score: {best_score_str}, counter: {self.counter}")

    if self.best_score is None or val_loss < self.best_score - self.min_delta:
      self.best_score = val_loss
      self.best_weigths = copy.deepcopy(model.state_dict())
      self.counter = 0
    else:
      self.counter += 1
    return self.counter >= self.patience # Return true iff should stop now

  def restore_best_weights(self, model):
    model.load_state_dict(self.best_weigths)

In [ ]:
# General training function with early stopping
def train_with_earlystop(model, optimizer, criterion, train_loader, valid_loader, n_epochs, patience=10, min_delta=0.001):
  early_stopping = EarlyStopping(patience=patience, min_delta=min_delta)
  for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      y_batch = y_batch.squeeze(1)

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    eval_loss = evaluate(model, valid_loader, criterion)
    if early_stopping.step(eval_loss, model):
      print("Early stopping triggered")
      early_stopping.restore_best_weights(model)
      break

    print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {mean_loss:.4f}, Validation Loss: {eval_loss:.4f}")

In [ ]:
model_pretrained_embeds = LieDetector_TextOnly_PretrainedEmbeds(bert_model.embeddings.word_embeddings).to(device)

learning_rate = 0.0005
optimizer = torch.optim.Adam(model_pretrained_embeds.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(weight=class_weights)

train_with_earlystop(model_pretrained_embeds,  optimizer=optimizer, criterion=criterion, train_loader = text_train_loader,
                     valid_loader = text_valid_loader, n_epochs=100)

To compare this model with our other baselines, let's compute its validation accuracy.

In [ ]:
metric = Accuracy(task="multiclass", num_classes=5).to(device)

In [ ]:
evaluate(model_pretrained_embeds, data_loader=text_valid_loader, criterion=metric)

This is pretty bad performance for the unbalanced dataset (where the majority classifier reaches 29% accuracy). It may be that the RNN model is suffering from the same issue as our metadata models, i.e. is it just degenerating to a majority classifier. To check, let's see its prediction counts.

In [ ]:
validation_results = sample_validation_results(model_pretrained_embeds, ds_valid, len(ds_valid))

validation_results.head()

In [ ]:
validation_results["Prediction"].value_counts()

We still have a collapse, although this time it is around half-true and barely-true rather than around false. This might be because false is the most common label, and so our weighted score assigns lower penalty for labeling false statements incorrectly. It makes sense to see how the model performs when we balance the dataset and don't rely on score weights.

#### 4.4.3 Rebalancing the dataset

In [ ]:
UnderSampler = RandomUnderSampler(random_state=42, replacement=True)

In [ ]:
X_under, y_under = UnderSampler.fit_resample(df_train_textonly[["Statement"]], df_train_textonly["Label"])

X_under_valid, y_under_valid = UnderSampler.fit_resample(df_valid_textonly[["Statement"]], df_valid_textonly["Label"])

y_under.value_counts(), y_under_valid.value_counts()

In [ ]:
df_train_textonly_under = pd.concat([X_under, y_under], axis=1)
df_valid_textonly_under = pd.concat([X_under_valid, y_under_valid], axis=1)

df_train_textonly_under.head()

In [ ]:
ds_train_under = Dataset.from_pandas(df_train_textonly_under)
ds_valid_under = Dataset.from_pandas(df_valid_textonly_under)

ds_train_under

In [ ]:
batch_size = 128
text_train_under_loader = DataLoader(ds_train_under, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
text_valid_under_loader = DataLoader(ds_valid_under, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

In [ ]:
model_pretrained_embeds = LieDetector_TextOnly_PretrainedEmbeds(bert_model.embeddings.word_embeddings).to(device)

learning_rate = 0.001
optimizer = torch.optim.Adam(model_pretrained_embeds.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

train_with_earlystop(model_pretrained_embeds,  optimizer=optimizer, criterion=criterion,
                     train_loader = text_train_under_loader, valid_loader = text_valid_under_loader, patience=20, n_epochs=100)

We see similar changes in the training scores, with initial successful learning being followed by overfitting.

In [ ]:
evaluate(model_pretrained_embeds, data_loader=text_valid_loader, criterion=metric)

The performance here is much better however, reaching over 32%. This is over 12% better than the baseline majority classifier, and 4% better than our best metadata-only baseline classifier on this rebalanced dataset.

In [ ]:
validation_results = sample_validation_results(model_pretrained_embeds, ds_valid_under, len(ds_valid_under))

validation_results.head()

In [ ]:
validation_results["Prediction"].value_counts()

It's very interesting to note that the model very rarely predicts the true label. A possible explanation is that true statements are linguistically very similar to almost-true statements, and the model is unable to distinguish between them effectively. So it just arbitrarily ends up chucking all such statements in the "almost-true" category. Let's look at the distribution of correct labels for "almost-true" predictions.

In [ ]:
precision = precision_score(validation_results["Label"], validation_results["Prediction"], average=None, zero_division=0),
recall = recall_score(validation_results["Label"], validation_results["Prediction"], average=None)

In [ ]:
print("precision: ", precision)
print("recall: ", recall)

This shows quite clearly that the model's performance on true statements differs considerably from its performance on other labels. The model achieves decent performance on false statements, middling performance for statements between true and false, with the model being overly careful when predicting true statements specifically.

#### 4.4.4. Using ordinal score

It's worth checking how the model performs when we train it with ordinal score rather than simple cross-entropy. To do this it helps to modify the model so that it outputs a single value

In [ ]:
class LieDetector_TextOnly_AsRegression(nn.Module):
  def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64, dropout=0.2):
    super().__init__()
    weights = pretrained_embeddings.weight.data

    self.embedding = nn.Embedding.from_pretrained(weights, freeze=True)
    embed_dim = weights.shape[-1]

    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)

    self.output = nn.Linear(hidden_dim, 1) # output a single value, that gets rounded to determine the class
  def forward(self, encodings):
    embeddings = self.embedding(encodings["input_ids"])
    _outputs, hidden_states = self.gru(embeddings)
    return self.output(hidden_states[-1])

In [ ]:
def evaluate_asregression(model, data_loader, criterion):
  model.eval()
  total_loss = 0
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)

      y_batch = y_batch.squeeze(1).float()

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
  return total_loss/len(data_loader)

In [ ]:
# General training function
def train_with_earlystop_asregression(model, optimizer, criterion, train_loader, valid_loader, n_epochs, patience=10, min_delta=0.001):
  early_stopping = EarlyStopping(patience=patience, min_delta=min_delta)
  for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      y_batch = y_batch.squeeze(1).float()

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    eval_loss = evaluate(model, valid_loader, criterion)
    if early_stopping.step(eval_loss, model):
      print("Early stopping triggered")
      early_stopping.restore_best_weights(model)
      break

    print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {mean_loss:.4f}, Validation Loss: {eval_loss:.4f}")

In [ ]:
model_as_regression = LieDetector_TextOnly_AsRegression(bert_model.embeddings.word_embeddings).to(device)

learning_rate = 0.0005
optimizer = torch.optim.Adam(model_as_regression.parameters(), lr=learning_rate)
criterion = nn.L1Loss()

train_with_earlystop(model_as_regression,  optimizer=optimizer, criterion=criterion,
                     train_loader = text_train_under_loader, valid_loader = text_valid_under_loader, patience=30, n_epochs=100)

After training the model as a regression model, we then evaluate it as a classifier by rounding the forecasts.

In [ ]:
def evaluate_asclassifier(model, data_loader, criterion):
  model.eval()
  total_loss = 0
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch).round().clamp(0,4).squeeze().long()
      y_batch = y_batch.squeeze(1).float()

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
  return total_loss/len(data_loader)

In [ ]:
def sample_validation_results_as_classifier(model, validation_set, sample_size):
  valid_length = len(validation_set)
  statement_list = list(validation_set["Statement"])
  label_list = list(validation_set["Label"])
  sample = random.sample(range(0, valid_length), sample_size)

  df = pd.DataFrame(columns=["Statement", "Label", "Prediction"])

  for entry_idx in sample:
    statement = statement_list[entry_idx]
    label = label_list[entry_idx]

    tokenized_statement = bert_tokenizer(statement, padding=True, truncation=True, max_length=80, return_tensors="pt")

    model.eval()
    with torch.no_grad():
      prediction = model(tokenized_statement.to(device)).cpu()
      pred_class = prediction.round().clamp(0,4).long().item()

      #print("prediction, ", prediction)
      #print("pred_class, ", pred_class)
      #print("label, ", label)
      df.loc[len(df)] = [statement, label, pred_class]
  return df

In [ ]:
evaluate_asclassifier(model_as_regression, data_loader=text_valid_loader, criterion=metric)

This is really bad, suggesting something may have gone very wrong with this model. And indeed that is the case, as shown by its prediction counts.

In [ ]:
validation_results = sample_validation_results_as_classifier(model_as_regression, ds_valid_under, len(ds_valid_under))

validation_results.groupby("Prediction").count()

The model has essentially collapsed to always predicting half-true, which hedges between all other options. This shows that the information available in the dataset is not sufficient to shift the model from just remaining maximally uncommitted in every case.

#### 4.4.5 Using pretrained contextualised embeddings

Instead of just using BERT's embeddings as inputs to our RNN, we can use BERT's entire last hidden state (i.e. the hidden state of its last layer) to provide us with a contextualized embedding for each token in the sentence, which we then feed to our RNN.

In [ ]:
class LieDetector_TextOnly_ContextualisedEmbeds(nn.Module):
  def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
    super().__init__()

    self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased").requires_grad_(False) # freeze BERT
    embed_dim = self.bert.config.hidden_size # the embeddings are just bert's hidden states

    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)

    self.output = nn.Linear(hidden_dim, 5) # output class logits

  def forward(self, encodings):
    contextualised_embeddings = self.bert(**encodings).last_hidden_state
    lengths = encodings["attention_mask"].sum(dim=1)
    packed = pack_padded_sequence(contextualised_embeddings, lengths=lengths.cpu(), batch_first=True, enforce_sorted=False)
    _outputs, hidden_states = self.gru(packed)
    return self.output(hidden_states[-1])

In [ ]:
model_contextualised_embeds = LieDetector_TextOnly_ContextualisedEmbeds(hidden_dim=128).to(device)


In [ ]:
learning_rate = 0.000005
optimizer = torch.optim.Adam(model_contextualised_embeds.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

train_with_earlystop(model_contextualised_embeds,  optimizer=optimizer, criterion=criterion,
                     train_loader = text_train_under_loader, valid_loader = text_valid_under_loader, patience=10, n_epochs=100)

This model performs similarly in terms of cross-entropy on validation set, but let's measure its accuracy.

In [ ]:
evaluate(model_contextualised_embeds, data_loader=text_valid_loader, criterion=metric)

This is worse than the model that simply used the BERT embeddings, rather than its contextualised embeddings! It suggests that perhaps the GRU layers might be the bottleneck to performance, being unable to effectively process all the information contained in BERT's contextualised embeddings.

To further test this conjecture, it's worth defining a model that directly passes the BERT outputs to a classification head, without using an RNN at all.

#### 4.4.6 Fine-tuning a full BERT model

We can fine-tune a pre-trained BERT model by using the Trainer API. This requires first defining tokenized datasets.

In [ ]:
def tokenize_batch(batch):
  return bert_tokenizer(batch["Statement"], padding=True, truncation=True, max_length=80)

In [ ]:
ds_train_under

In [ ]:
tokenized_train_ds = ds_train_under.map(tokenize_batch, batched=True).rename_column("Label", "labels")
tokenized_valid_ds = ds_valid_under.map(tokenize_batch, batched=True).rename_column("Label", "labels")

In [ ]:
tokenized_valid_ds

we define a new accuracy function for use by the Trainer API

In [ ]:
def compute_accuracy(pred):
  return {"accuracy": (pred.predictions.argmax(axis=-1) == pred.label_ids).mean()}

Next we define our training configuration

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/myBERTmodel"

In [ ]:
train_args = TrainingArguments(
    output_dir = OUTPUT_PATH,
    num_train_epochs = 10,
    per_device_train_batch_size = 128,
    per_device_eval_batch_size = 128,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    logging_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "accuracy",
    report_to = "none"
)

We now create the model to be trained: a BERT pretrained model with a (untrained) classifier head on top.

In [ ]:
torch.manual_seed(42)

bert_classifier = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=5, dtype=torch.float32).to(device)

Then we create a Trainer object and pass it the model and training arguments, as well as the datasets and the evaluation function, plus a data collator to take care of padding.

In [ ]:
trainer = Trainer(
    bert_classifier,
    train_args,
    train_dataset = tokenized_train_ds,
    eval_dataset = tokenized_valid_ds,
    compute_metrics = compute_accuracy,
    data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)
)

In [ ]:
train_output = trainer.train()

This does noticeably worse than the model reusing BERT pretrained embeddings, and a little better than the model using contextualised BERT embeddings as inputs to an RNN. It starts overfitting heavily after only a few epochs.

Plausibly these three models are all performing very similarly. The dataset may just not be informative enough for their different complexity to make much of a difference here. Indeed, the validation is small enough that validation perofrmance will have high variance, so these three models could well be generalizing about as well as one another. In the next section we will look at augmenting the base text data with context data and metadata, and see whether that makes any meaningful difference in the model's performance.

## 5. Main Model: Text, context, and metadata classifier

### 5.1 Preparing the data

#### 5.1.1 Preprocessing Categorical Features and Labels

The model will learn from both text and categorical data simultaneously. For the categorical data, Speaker and Party will be encoded via a trainable embedding layer. Subject will be encoded via average pooling of embeddings: each subject value that is present for a sample goes through a trainable embedding layer, and then the embeddings of all applicable subjects will be averaged to produce a single embedding that summarizes the sample subject. To make these transformations a bit simpler, it helps to modify our preprocessed dataframe to make subject into a single list-valued column.

In [ ]:
df_train_main = df_train_preprocessed.copy()
df_valid_main = df_valid_preprocessed.copy()

In [ ]:
df_train_main["Subjects"] = df_train_main.apply(lambda row: [col for col in list(common_subjects) + ["other"] if row[col] ==1], axis=1)
df_valid_main["Subjects"] = df_valid_main.apply(lambda row: [col for col in list(common_subjects) + ["other"] if row[col] ==1], axis=1)

In [ ]:
df_train_main["Subjects"][:10]

Then we map all categorical features and subjects into vectors for the embedding layer. To begin with, we map each subject to an index number. Then, for each sample, we will collect all its indices into a list, and finally map the list to a vector of fixed length.

In [ ]:
common_subject_ordered = ["other"] + list(common_subjects)

subject_to_id_dictionary = {subject: idx for idx, subject in enumerate(common_subject_ordered)}

def subject_to_id(subject):
  if subject in common_subjects:
    return subject_to_id_dictionary[subject]
  else:
    return subject_to_id_dictionary["other"]

In [ ]:
subject_to_id("other"), subject_to_id("elections"), subject_to_id("never-before-seen-subject")

In [ ]:
party_counts = df_train_main["Party"].value_counts()
common_parties_ordered = ["other"] + list(party_counts[party_counts>25].index)
common_parties_ordered

In [ ]:
party_to_id_dictionary = {party: idx for idx, party in enumerate(common_parties_ordered)}

def party_to_id(party):
  if party in common_parties_ordered:
    return party_to_id_dictionary[party]
  else:
    return party_to_id_dictionary["other"]

In [ ]:
speaker_counts = df_train_main["Speaker"].value_counts()
common_speakers_ordered = ["other"] + list(speaker_counts[speaker_counts>25].index)

In [ ]:
speaker_to_id_dictionary = {speaker: idx for idx, speaker in enumerate(common_speakers_ordered)}

def speaker_to_id(speaker):
  if speaker in common_speakers_ordered:
    return speaker_to_id_dictionary[speaker]
  else:
    return speaker_to_id_dictionary["other"]

In [ ]:
df_train_main["Party_id"] = df_train_main["Party"].apply(party_to_id)
df_train_main["Speaker_id"] = df_train_main["Speaker"].apply(speaker_to_id)
df_train_main["Subject_ids"] = df_train_main["Subjects"].apply(lambda subjects: [subject_to_id(subject) for subject in subjects])

df_valid_main["Party_id"] = df_valid_main["Party"].apply(party_to_id)
df_valid_main["Speaker_id"] = df_valid_main["Speaker"].apply(speaker_to_id)
df_valid_main["Subject_ids"] = df_valid_main["Subjects"].apply(lambda subjects: [subject_to_id(subject) for subject in subjects])

In [ ]:
df_train_main[["Party_id", "Speaker_id", "Subject_ids"]].head()

The final step is to convert Subject_ids from lists of indices to vectors of fixed length. The idea is that list `[x_1, ..., x_n]` gets mapped to vector `[k_1, ..., k_N]` such that:

$$ k_j = \begin{cases}
 1 \text{ if } j \in \{x_1, ..., x_n\} \\
 0 \text{ otherwise }
\end{cases} $$

E.g. list `[5, 32]` gets mapped to a vector with value 1 on dimensions 5 and 32, and value 0 on all other dimensions. The length of these vectors is given by `len(common_subject_ordered)`

In [ ]:
def subject_list_to_vector(subject_list):
  vector = [0]*len(common_subject_ordered)
  for i in subject_list:
    vector[i] = 1
  return vector

torch.Tensor(subject_list_to_vector([5, 32]))

In [ ]:
df_train_main["Subject_ids"] = df_train_main["Subject_ids"].apply(subject_list_to_vector)
df_valid_main["Subject_ids"] = df_valid_main["Subject_ids"].apply(subject_list_to_vector)

Now we map the labels to indices.

In [ ]:
label_to_int = {"false": 0, "barely-true": 1, "half-true": 2, "mostly-true": 3, "true": 4}

df_train_main["Label"] = df_train_main["Label"].apply(lambda label: label_to_int[label])
df_valid_main["Label"] = df_valid_main["Label"].apply(lambda label: label_to_int[label])

In [ ]:
df_train_main["Label"].value_counts()

And finally, use undersampling to balance the classes.

In [ ]:
UnderSampler = RandomUnderSampler(random_state=42, replacement=True)

X_train_main, y_train_main = UnderSampler.fit_resample(df_train_main.drop("Label", axis=1), df_train_main["Label"])

X_valid_main, y_valid_main = UnderSampler.fit_resample(df_valid_main.drop("Label", axis=1), df_valid_main["Label"])

In [ ]:
y_train_main.value_counts(), y_valid_main.value_counts()

In [ ]:
df_train_main = pd.concat([X_train_main, y_train_main], axis=1)
df_valid_main = pd.concat([X_valid_main, y_valid_main], axis=1)

df_train_main.head()

#### 5.1.2 Preprocessing text data

We want to feed our RNN both the main statement and the context, so we will need to append these two strings, separating them with a [SEP] token. E.g.

`sentence = "[CLS] Sentence A [SEP] Sentence B [SEP]"`

Furthermore, since we might want to use a BERT model for contextualised embeddings, or for direct cassification, we will also keep a parallel index of token_type_ids. This index shows to which of the two sentences each token belongs.



`token_type_ids = [0, 0, 0, 0, 0, 1, 1, 1, 1]`

This can all be done at the data-loader level by applying the bert tokenizer in the collate function. However it is more convenient to do this at the dataset level, as tokenized datasets are necessary for using the Trainer API in case we decide to fine-tune a full pretrained BERT model.

Let's start by first converting all text data to lowercase.

In [ ]:
df_train_main["Statement"] = df_train_main["Statement"].apply(lambda statement: statement.lower())
df_train_main["Context"] = df_train_main["Context"].apply(lambda statement: statement.lower())

df_valid_main["Statement"] = df_valid_main["Statement"].apply(lambda statement: statement.lower())
df_valid_main["Context"] = df_valid_main["Context"].apply(lambda statement: statement.lower())

In [ ]:
df_train_main[["Statement", "Context"]].head()

Then we remove unnecessary columns and convert to datasets.

In [ ]:
df_train_main = df_train_main[["Statement", "Context", "Party_id", "Speaker_id", "Subject_ids", "Label"]]
df_valid_main = df_valid_main[["Statement", "Context", "Party_id", "Speaker_id", "Subject_ids", "Label"]]


df_train_main.head()

In [ ]:
ds_train_main = Dataset.from_pandas(df_train_main)
ds_valid_main = Dataset.from_pandas(df_valid_main)

In [ ]:
ds_train_main

Now we apply the Bert tokenizer to both text sequences to generate their tokenised concatenation, including token_type_ids.

In [ ]:
print(ds_train_main[0])
print(type(ds_train_main[0]["Statement"]))
print(type(ds_train_main[0]["Context"]))


In [ ]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize_batch_multisequence(batch):
  # leave padding for later using DataCollatorWithPadding
  return bert_tokenizer(batch["Statement"], batch["Context"], padding=False, truncation=True, max_length=80)

In [ ]:
ds_train_main_tokenized = (ds_train_main.map(tokenize_batch_multisequence, batched=True)
  .rename_column("Label", "labels")
  .remove_columns(["Statement", "Context"]))
ds_valid_main_tokenized = (ds_valid_main.map(tokenize_batch_multisequence, batched=True)
  .rename_column("Label", "labels")
  .remove_columns(["Statement", "Context"]))


ds_train_main_tokenized

In [ ]:
ds_valid_main["Statement"][4], ds_valid_main["Context"][4]

Finally we create data loaders to be used by models that are not trained via the Trainer API.

In [ ]:
class CollatorClass:
  def __init__(self, tokenizer):
    self.collator = DataCollatorWithPadding(tokenizer=tokenizer)
  def __call__(self, batch):
    embeddings = self.collator(batch)
    labels = embeddings.pop("labels")
    return embeddings, labels

In [ ]:
batch_size = 128
collator = CollatorClass(bert_tokenizer)

train_main_loader = DataLoader(ds_train_main_tokenized, batch_size=batch_size, shuffle=True, collate_fn=collator)
valid_main_loader = DataLoader(ds_valid_main_tokenized, batch_size=batch_size, shuffle=False, collate_fn=collator)

In [ ]:
# DEBUG
#for X_batch, y_batch in train_main_loader:
#  print(X_batch["input_ids"].shape)
#  print(X_batch["Party_id"].shape)
#  print(X_batch["Speaker_id"].shape)
#  print(X_batch["Subject_ids"].shape)
#  print(y_batch.shape)
#  break

### 5.2 Building the Models

In [ ]:
if torch.cuda.is_available():
    device="cuda"
elif torch.backends.mps.is_available:
    device="mps"
else:
    device="cpu"

device

In [ ]:
torch.manual_seed(42)

### 5.2.1 Pretrained Embeddings

Given how poorly a model trained from scratch performed in the previous section (4.4.1), it seems reasonable to start by looking at models which already inherit their embedding layer from a pretrained BERT model.

In [ ]:
class LieDetector_Full_PretrainedEmbeds(nn.Module):
  def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64, pad_id=0, dropout=0.2):
    super().__init__()

    # Text processing components
    weights = pretrained_embeddings.weight.data
    self.embedding_text = nn.Embedding.from_pretrained(weights, freeze=True)
    embed_dim_text = weights.shape[-1]
    self.gru = nn.GRU(embed_dim_text, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)

    # Metadata processing components
    self.embedding_party = nn.Embedding(len(common_parties_ordered), 16, padding_idx=pad_id)
    self.embedding_speaker = nn.Embedding(len(common_speakers_ordered), 16, padding_idx=pad_id)
    self.embedding_subject = nn.Embedding(len(common_subject_ordered), 32, padding_idx=pad_id)

    total_size = hidden_dim + 16 + 16 + 32
    self.output = nn.Linear(total_size, 5) # output class logits

  def forward(self, X_batch):
    # Text processing
    text_embeddings = self.embedding_text(X_batch["input_ids"])
    _outputs, hidden_states = self.gru(text_embeddings)

    # Metadata processing
    party_embeddings = self.embedding_party(X_batch["Party_id"])
    speaker_embeddings = self.embedding_speaker(X_batch["Speaker_id"])

    # Trick for efficiency: to apply embedding layer to the indices of the multihot
    # vector we can just multiply the multihot vector by the weights of the embedding layer
    subject_sum = X_batch["Subject_ids"].float() @ self.embedding_subject.weight
    counts = X_batch["Subject_ids"].sum(dim=1, keepdim=True).clamp(min=1)
    subject_embeddings = subject_sum / counts

    return self.output(torch.cat([hidden_states[-1], party_embeddings, speaker_embeddings, subject_embeddings], dim=1))

In [ ]:
model_path = "/content/drive/MyDrive/models/bert-base-uncased"

if not os.path.exists(model_path):
    bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")
    bert_model.save_pretrained(model_path)
else:
    bert_model = transformers.AutoModel.from_pretrained(model_path)

In [ ]:
model_pretrained_embeds = LieDetector_Full_PretrainedEmbeds(bert_model.embeddings.word_embeddings).to(device)
model_pretrained_embeds

In [ ]:
# Evaluation function (not using torchmetrics)
def evaluate(model, data_loader, criterion):
  model.eval()
  total_loss = 0
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)

      #y_batch = y_batch.squeeze(1)

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
  return total_loss/len(data_loader)

In [ ]:
# General training function
def train_with_earlystop(model, optimizer, criterion, train_loader, valid_loader, n_epochs, patience=10, min_delta=0.001):
  early_stopping = EarlyStopping(patience=patience, min_delta=min_delta)
  for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)

     # print(X_batch["input_ids"].shape)
     # print(y_batch.shape)
     # print(y_pred.shape)

     #y_batch = y_batch.squeeze(1)

      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    eval_loss = evaluate(model, valid_loader, criterion)
    if early_stopping.step(eval_loss, model):
      print("Early stopping triggered")
      early_stopping.restore_best_weights(model)
      break

    print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {mean_loss:.4f}, Validation Loss: {eval_loss:.4f}")

In [ ]:
learning_rate = 0.0001
optimizer = torch.optim.Adam(model_pretrained_embeds.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [ ]:
train_with_earlystop(model_pretrained_embeds,  optimizer=optimizer, criterion=criterion,
                     train_loader = train_main_loader, valid_loader = valid_main_loader,
                     patience=20, n_epochs=200)

In [ ]:
metric = Accuracy(task="multiclass", num_classes=5).to(device)

In [ ]:
evaluate(model_pretrained_embeds, data_loader=valid_main_loader, criterion=metric)

This is  slightly worse than our baseline model using Bert embeddings on the rebalanced dataset.

One cause could be that the context information is just adding noise to the data, and worsening the model's performance. After all, we have seen that both main statement text and metadata, on their own, are at least somewhat informative about the truthfulness of the claim. So the context text is a natural suspect here. Let's see how the model performs without context.

#### 5.2.1.1 Removing context

In [ ]:
def tokenize_batch_singlesequence(batch):
  # leave padding for later using DataCollatorWithPadding
  return bert_tokenizer(batch["Statement"], padding=False, truncation=True, max_length=80)

In [ ]:
ds_train_nocontext_tokenized = (ds_train_main.map(tokenize_batch_singlesequence, batched=True)
  .rename_column("Label", "labels")
  .remove_columns(["Statement", "Context"]))
ds_valid_nocontext_tokenized = (ds_valid_main.map(tokenize_batch_singlesequence, batched=True)
  .rename_column("Label", "labels")
  .remove_columns(["Statement", "Context"]))

In [ ]:
batch_size = 128
collator = CollatorClass(bert_tokenizer)

train_nocontext_loader = DataLoader(ds_train_nocontext_tokenized, batch_size=batch_size, shuffle=True, collate_fn=collator)
valid_nocontext_loader = DataLoader(ds_valid_nocontext_tokenized, batch_size=batch_size, shuffle=False, collate_fn=collator)

In [ ]:
model_pretrained_embeds_nocontext = LieDetector_Full_PretrainedEmbeds(bert_model.embeddings.word_embeddings).to(device)
model_pretrained_embeds_nocontext

In [ ]:
learning_rate = 0.0001
optimizer = torch.optim.Adam(model_pretrained_embeds_nocontext.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [ ]:
train_with_earlystop(model_pretrained_embeds_nocontext,  optimizer=optimizer, criterion=criterion,
                     train_loader = train_nocontext_loader, valid_loader = valid_nocontext_loader,
                     patience=20, n_epochs=200)

This appears to do better in terms of cross-entropy. Let's see how it does in terms of accuracy.

In [ ]:
metric = Accuracy(task="multiclass", num_classes=5).to(device)

In [ ]:
evaluate(model_pretrained_embeds_nocontext, data_loader=valid_nocontext_loader, criterion=metric)

Depending on the seed and on inherently non-deterministic CUDA operations, this can do slightly better or slightly worse than our model with context.

More generally, we can conclude that for models using BERT pretrained embeddings, multiple feature combinations (text only, text+metadata, text+context+metadata) reach accuracy in the 29-31% range. No combination of features shows a consistent advantage, suggesting that BERT-derived text representations capture most of the available signal and additional metadata provides limited incremental value.

In [ ]:
## This is not at all optimized for large datasets, should be used only to test out
## model on small samples.
def sample_validation_results(model, validation_set, sample_size):
  valid_length = len(validation_set)
  input_list = list(validation_set["input_ids"])
  label_list = list(validation_set["labels"])
  sample = random.sample(range(0, valid_length), sample_size)

  df = pd.DataFrame(columns=["Statement", "Label", "Prediction"])

  for entry_idx in sample:
    tokenized_statement = torch.tensor(input_list[entry_idx], dtype=torch.long)

    decoded_statement = bert_tokenizer.decode(tokenized_statement)

    label = label_list[entry_idx]

    entry = validation_set[entry_idx]
    entry["input_ids"] = torch.tensor([entry["input_ids"]],  dtype=torch.int32).to(device)
    entry["Party_id"] = torch.tensor([entry["Party_id"]], dtype=torch.int32).to(device)
    entry["Speaker_id"] = torch.tensor([entry["Speaker_id"]], dtype=torch.int32).to(device)
    entry["Subject_ids"] = torch.tensor([entry["Subject_ids"]], dtype=torch.int32).to(device)

    model.eval()

    with torch.no_grad():
      prediction = model(entry).argmax(dim=-1).item()
      df.loc[len(df)] = [decoded_statement, label, prediction]
  return df

In [ ]:
sample_validation_results(model_pretrained_embeds_nocontext, ds_valid_nocontext_tokenized,
                          sample_size=len(ds_valid_nocontext_tokenized))["Prediction"].value_counts().sort_index()

  Interestingly, this is far less bunched-up than any of our previous models, although the pattern of labeling few inputs as "true" observed in previous models is still present. This further suggests that the model is learning something about the dataset rather than degenerating to a majority classifier, and indeed this model is considerably more accurate than such a classifier.